<a href="https://colab.research.google.com/github/vnandigam/CHM_lidar_lidR/blob/main/canopy_height_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Canopy Height Maps from fusion of NASA GEDI, Sentinel-1 RTC SAR, Sentinel-2 L2A Optical and COP30 DEM

This notebook, designed to run in **Google Colab**, builds a wall-to-wall forest canopy height map by combining freely available open-access satellite and lidar datasets. Canopy height is a key structural attribute of forests - it correlates with aboveground biomass, carbon stocks, and biodiversity - but field campaigns to measure it are expensive and spatially limited. By pairing NASA GEDI's sparse but accurate lidar footprints with the dense spatial coverage of Sentinel-2 optical imagery, Sentinel-1 RTC SAR imagery, and the Copernicus 30 m digital elevation model, this workflow trains a Random Forest to extrapolate height estimates across any user-defined area at 30 m resolution. Three experiments are run in sequence, each adding a new data source, so users can directly compare how much each layer improves prediction accuracy.

| Source Data | What it provides |
|---|---|
| **GEDI L2A** (NASA) | Ground-truth canopy height (`rh98`) from spaceborne lidar |
| **Sentinel-2 L2A** (ESA, via Microsoft Planetary Computer) | 10-band optical imagery + NDVI |
| **Sentinel-1 SAR** (ESA, via Microsoft Planetary Computer) | VV, VH backscatter - sees through clouds |
| **COP30 DEM** (Copernicus, via OpenTopography) | Elevation, slope, aspect at 30 m |

**Workflow:**
- Define a spatial area of interest/bounding box
- Download summer-2024 GEDI footprints inside the bbox
- Build a Sentinel-2 cloud-masked seasonal median composite
- **Experiment 1** — Train a Random Forest using Sentinel-2 spectral features only
- **Experiment 2** — Add Sentinel-1 SAR features and re-evaluate
- **Experiment 3** — Add COP30 topography and re-evaluate
- Compare R² / RMSE across experiments and export canopy height GeoTIFFs

Authors: Kai Lin, Viswanath Nandigam  
San Diego Supercomputer Center, University of California San Diego    
Work is part of OpenForest4D project, funded by the U.S. National Science Foundation (NSF) under awards 2409885, 2409886 & 2409887


## 1. Setup

Install and import the geospatial and ML libraries required by this workflow. Some packages (earthaccess, planetary-computer, leafmap) are not present in default Colab images and must be installed before importing.

In [ ]:
# If running locally, ensure you have activated your conda/virtual environment before launching Jupyter.
# If running in Google Colab, execute the cell below to set up the necessary third-party libraries.
!pip install -q fsspec==2025.3.0 earthaccess h5py pystac-client planetary-computer rasterio leafmap scikit-learn matplotlib

In [ ]:
import os
import io
import re
import time
import math
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.windows import from_bounds as window_from_bounds
from rasterio.transform import from_bounds as transform_from_bounds
from pyproj import Transformer
from scipy.ndimage import zoom

import pystac_client
import planetary_computer # signs Azure Blob SAS tokens for Planetary Computer assets
import earthaccess # NASA EarthData authentication + granule search/download

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings("ignore", category=UserWarning)

WORK_DIR = Path("./canopy_workflow")
WORK_DIR.mkdir(exist_ok=True)
GEDI_DIR = WORK_DIR / "gedi"; GEDI_DIR.mkdir(exist_ok=True)
print(f"Working directory: {WORK_DIR.resolve()}")


## 2. Area of interest

Define the spatial extent of the analysis as a WGS84 bounding box. The example uses the western Oregon Cascades — a temperate conifer landscape with sharp topographic relief. Keep the bounding box small (~30 × 30 km) to stay within Colab's memory limits; the cell prints an approximate area in km² as a sanity check.

In [ ]:
# bbox = [min_lon, min_lat, max_lon, max_lat]   (WGS84)
bbox = [-122.85, 43.28, -122.50, 43.55]

# crude size estimate
lat_c = 0.5 * (bbox[1] + bbox[3])
w_km = (bbox[2] - bbox[0]) * 111.0 * math.cos(math.radians(lat_c))
h_km = (bbox[3] - bbox[1]) * 111.0
print(f"BBox: {bbox}")
print(f"Approximate size: {w_km:.1f} × {h_km:.1f} km  (~{w_km*h_km:.0f} km²)")


In [ ]:
# Map preview (optional). Uses leafmap; in Colab the cell renders inline.

import leafmap
m = leafmap.Map(center=[(bbox[1]+bbox[3])/2, (bbox[0]+bbox[2])/2],
                zoom=10, height="400px", width="700px")
m.add_basemap("CartoDB.Positron")     # clean, light grey
m.add_basemap("CartoDB.Voyager")      # colored, modern
m.add_basemap("Esri.WorldStreetMap")  # Esri street map
m.add_geojson({
    "type": "Feature", "properties": {},
    "geometry": {"type": "Polygon",
                  "coordinates": [[[bbox[0],bbox[1]], [bbox[2],bbox[1]],
                                  [bbox[2],bbox[3]], [bbox[0],bbox[3]],
                                  [bbox[0],bbox[1]]]]}
}, layer_name="AOI")
m


## 3. Download NASA GEDI L2A footprints (Summer 2024)

Authenticate with NASA EarthData (`earthaccess`) and download GEDI L2A HDF5 granules that overlap the bounding box for June–September 2024. For each granule, full-power beam shots are filtered to those passing quality and degrade flags, and the `rh98` metric (98th-percentile relative height — a standard proxy for canopy top height) is extracted. Shots with implausible heights (<0 m or >80 m) are dropped.

This requires a free [NASA EarthData account](https://urs.earthdata.nasa.gov/users/new).
Fill in your credentials in the next cell before running.


### Optional: Secure Credential Handling via Colab Secrets

If you are running this notebook in **Google Colab**, you can store your credentials as [Colab Secrets](https://medium.com/@parthdasawant/how-to-use-secrets-in-google-colab-450c38e3ec75) instead of pasting them directly into the cell below. This prevents your username and password from appearing in the notebook file or version history.

**To set up Colab Secrets:**
1. Click the **🔑 key icon** in the left sidebar (or go to **Tools → Secrets**).
2. Add two secrets:
   - `EARTHDATA_USERNAME` → your NASA EarthData username
   - `EARTHDATA_PASSWORD` → your NASA EarthData password
3. Toggle **Notebook access** on for each secret.
4. Run the cell below — it will automatically read from Secrets if available, and fall back to the hardcoded values otherwise.

> **Note:** If you are running locally (Jupyter, VS Code, etc.), skip this step and fill in your credentials directly in the next cell.

In [ ]:
# ── Secure Credential Handling via Colab Secrets (optional) ──────────────────
# When running in Google Colab, credentials are read from the Secrets store
# so they never appear in plain text in the notebook.
# When running locally, this block is silently skipped and the variables
# defined in the next cell are used instead.

try:
    from google.colab import userdata  # available only inside Colab
    _ed_user = userdata.get("EARTHDATA_USERNAME")
    _ed_pass = userdata.get("EARTHDATA_PASSWORD")
    if _ed_user and _ed_pass:
        EARTHDATA_USERNAME = _ed_user
        EARTHDATA_PASSWORD = _ed_pass
        print("✓ EarthData credentials loaded from Colab Secrets.")
    else:
        print("⚠ Colab Secrets found but EARTHDATA_USERNAME / EARTHDATA_PASSWORD "
              "are empty. Falling back to the values set in the next cell.")
        EARTHDATA_USERNAME = None
        EARTHDATA_PASSWORD = None
except Exception:
    # Not running in Colab, or the secret keys don't exist yet.
    EARTHDATA_USERNAME = None
    EARTHDATA_PASSWORD = None

In [ ]:
# ── Hardcoded fallback credentials ───────────────────────────────────────────
# Fill these in if you are NOT using Colab Secrets (see the cell above).
# If Colab Secrets were loaded successfully, these lines are skipped.

if EARTHDATA_USERNAME is None:
    EARTHDATA_USERNAME = "USERNAME"    # e.g. "yourusername"
if EARTHDATA_PASSWORD is None:
    EARTHDATA_PASSWORD = "PASSWORD"    # e.g. "yourpassword"

assert EARTHDATA_USERNAME not in (None, "USERNAME") and \
       EARTHDATA_PASSWORD not in (None, "PASSWORD"), (
    "Please set EARTHDATA_USERNAME and EARTHDATA_PASSWORD before running — "
    "either via Colab Secrets (recommended) or by editing this cell directly."
)
os.environ["EARTHDATA_USERNAME"] = EARTHDATA_USERNAME
os.environ["EARTHDATA_PASSWORD"] = EARTHDATA_PASSWORD
print("EarthData credentials set.")

In [ ]:
# Authenticate using the credentials above.
earthaccess.login(strategy="environment")

results = earthaccess.search_data(
    short_name="GEDI02_A",
    bounding_box=tuple(bbox),
    temporal=("2024-06-01", "2024-09-30"),
)
print(f"Granules matched: {len(results)}")

# Download the HDF5 files (cached in WORK_DIR/gedi)
files = earthaccess.download(results, str(GEDI_DIR))
files = [str(f) for f in files if f and str(f).endswith(".h5")]
print(f"Downloaded: {len(files)} files")


In [ ]:
# Only full-power beams are used. GEDI has 8 beams total: 4 coverage (lower
# energy) and 4 full-power. Full-power beams have higher sensitivity in dense
# canopy and produce more reliable rh98 estimates.

POWER_BEAMS = ["BEAM0101","BEAM0110","BEAM1000","BEAM1011"]   # full-power beams

def _gedi_shots_from_file(h5_path, bbox):
    """Extract per-shot rh98 + lat/lon for shots inside bbox passing quality filters.
    Filters applied:
      - quality_flag == 1: shot meets GEDI's basic quality criteria
      - degrade_flag == 0: no known data quality degradation (e.g. pointing errors)
      - bbox spatial intersection
    rh98 is the 98th-percentile relative height above ground — the standard
    proxy for canopy top height in GEDI L2A products.
    """
    rows = []
    minlon, minlat, maxlon, maxlat = bbox
    with h5py.File(h5_path, "r") as f:
        for beam in f.keys():
            if beam not in POWER_BEAMS:
                continue
            try:
                g = f[beam]
                lon  = g["lon_lowestmode"][:]   # longitude of lowest energy mode
                lat  = g["lat_lowestmode"][:]   # latitude  of lowest energy mode
                qf   = g["quality_flag"][:]     # 1 = good quality
                deg  = g["degrade_flag"][:]     # 0 = no degradation
                sens = g["sensitivity"][:]      # fraction of canopy cover detectable
                rh   = g["rh"][:]               # relative height percentiles, shape (N, 101)
                shot = g["shot_number"][:]
            except KeyError:
                continue

            # Boolean mask: shots inside bbox AND passing both quality flags
            keep = ((lon >= minlon) & (lon <= maxlon) &
                    (lat >= minlat) & (lat <= maxlat) &
                    (qf == 1) & (deg == 0))
            if not keep.any():
                continue
            sub = np.where(keep)[0]
            for i in sub:
                rows.append({
                    "beam": beam, "shot_number": int(shot[i]),
                    "longitude": float(lon[i]), "latitude": float(lat[i]),
                    "sensitivity": float(sens[i]),
                    "rh98": float(rh[i, 98]),   # index 98 = 98th percentile
                })
    return rows

records = []
for fp in files:
    try:
        records.extend(_gedi_shots_from_file(fp, bbox))
    except Exception as e:
        print(f"  ⚠ Skipping {Path(fp).name}: {e}")

gedi = pd.DataFrame(records)
# Drop physically implausible heights: negatives indicate ground returns or
# noise; >80 m exceeds realistic temperate forest canopy heights.
gedi = gedi[(gedi["rh98"] > 0) & (gedi["rh98"] < 80)].reset_index(drop=True)
gedi.to_csv(WORK_DIR / "gedi_shots.csv", index=False)
print(f"GEDI shots retained: {len(gedi):,}")
gedi.head()


## 4. Sentinel-2 L2A — cloud-masked summer median composite

Query Microsoft Planetary Computer for Sentinel-2 L2A scenes with less than 20% cloud cover (`eo:cloud_cover < 20%`) over the bounding box during summer 2024. Each scene is cloud-masked using the SCL (Scene Classification Layer), and a pixel-wise **median** is computed across all valid observations to produce a single clean 11-band composite (10 spectral bands + NDVI) at 30 m resolution. Taking the median rather than a single scene removes residual cloud shadows and atmospheric noise.

In [ ]:
S2_BANDS = ["B02","B03","B04","B05","B06","B07","B08","B8A","B11","B12"]
S2_RES  = 30  # output resolution in metres; native S2 bands range from 10–60 m

cat = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
# --- STAC search ---
search = cat.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2024-06-01/2024-09-30",
    query={"eo:cloud_cover": {"lt": 20}},
)
items = list(search.items())
print(f"Sentinel-2 scenes matched: {len(items)}")
# Sort by cloud cover and keep the 10 clearest to limit download volume
items = sorted(items, key=lambda it: it.properties["eo:cloud_cover"])[:10]
print(f"Using {len(items)} clearest scenes")

# --- EPSG resolution ---
# STAC spec changed `proj:epsg` (v1) to `proj:code` (v2); try both to stay
# compatible with older and newer catalog responses.
def _epsg_of(item):
    p = item.properties
    if "proj:epsg" in p and p["proj:epsg"] is not None:
        return int(p["proj:epsg"])
    if "proj:code" in p and p["proj:code"]:
        return int(str(p["proj:code"]).split(":")[-1])
    # Fall back to asset-level fields
    for a in item.assets.values():
        ef = getattr(a, "extra_fields", {}) or {}
        if "proj:epsg" in ef and ef["proj:epsg"] is not None:
            return int(ef["proj:epsg"])
        if "proj:code" in ef and ef["proj:code"]:
            return int(str(ef["proj:code"]).split(":")[-1])
    # Last resort: open the file and read CRS directly
    for a in item.assets.values():
        try:
            with rasterio.open(a.href) as src:
                if src.crs and src.crs.to_epsg():
                    return int(src.crs.to_epsg())
        except Exception:
            continue
    raise RuntimeError("Could not determine EPSG for STAC item")

# Define the output grid in the scene's native UTM CRS at S2_RES resolution.
# Snapping width/height to whole pixels ensures a clean grid alignment.
crs0 = _epsg_of(items[0])
left, bottom, right, top = transform_bounds("EPSG:4326", f"EPSG:{crs0}", *bbox)
w = int(math.ceil((right - left) / S2_RES))
h = int(math.ceil((top  - bottom) / S2_RES))
right  = left + w * S2_RES # snap to pixel boundary
bottom = top  - h * S2_RES
dst_tfm = transform_from_bounds(left, bottom, right, top, w, h)
print(f"Output grid: {h} × {w}  CRS=EPSG:{crs0}  res={S2_RES} m")

def _read_band(item, asset_name, dst_tfm, dst_crs, h, w, retries=3):
    """Read one band and reproject it to the target grid.

    Re-signs the SAS token on every attempt because Planetary Computer tokens
    expire; exponential backoff handles transient HTTP 503/403 errors.
    """
    raw_href = item.assets[asset_name].href # refresh SAS token
    out = np.full((h, w), np.nan, dtype="float32")
    last_exc = None
    for attempt in range(retries):
        try:
            href = planetary_computer.sign(raw_href)
            with rasterio.open(href) as src:
                reproject(
                    source=rasterio.band(src, 1), destination=out,
                    src_transform=src.transform, src_crs=src.crs,
                    dst_transform=dst_tfm, dst_crs=f"EPSG:{crs0}",
                    resampling=Resampling.bilinear,
                )
            return out
        except Exception as e:
            last_exc = e
            wait = 2 ** attempt   # 1, 2, 4 s
            print(f"    retry {attempt+1}/{retries} for {asset_name} "
                  f"(waiting {wait}s): {type(e).__name__}: {str(e)[:60]}")
            time.sleep(wait)
    raise last_exc

def _scl_mask(item, dst_tfm, h, w):
    """Return a boolean keep-mask from the Sentinel-2 Scene Classification Layer.

    Accepted SCL classes (pixels treated as valid):
      4 = vegetation, 5 = bare soil, 6 = water, 7 = unclassified (low cloud prob),
      11 = snow/ice
    Rejected: 1-2 (saturated/defective), 3 (cloud shadow), 8-10 (cloud)
    """
    arr = _read_band(item, "SCL", dst_tfm, f"EPSG:{crs0}", h, w)
    return np.isin(arr.astype(np.int16), [4, 5, 6, 7, 11])

print("Reading bands + cloud-masking each scene...")
band_stacks = {b: [] for b in S2_BANDS}
for k, it in enumerate(items, 1):
    try:
        ok = _scl_mask(it, dst_tfm, h, w)
    except Exception as e:
        print(f"  ⚠ scene {k} SCL unreadable, skipping scene: {e}")
        continue
    for b in S2_BANDS:
        try:
            arr = _read_band(it, b, dst_tfm, f"EPSG:{crs0}", h, w)
            arr[~ok] = np.nan # mask cloud/shadow pixels before stacking
            band_stacks[b].append(arr)
        except Exception as e:
            print(f"  ⚠ scene {k} band {b}: {e}")
    print(f"  scene {k}/{len(items)} done")

# Pixel-wise nanmedian collapses the scene stack into a single clean composite.
# nanmedian ignores NaN (masked) pixels so cloud gaps in one scene are filled
# by clear observations from other scenes.
composite = np.stack(
    [np.nanmedian(np.stack(band_stacks[b], axis=0), axis=0) for b in S2_BANDS],
    axis=0,
).astype("float32")

# NDVI = (B08 - B04) / (B08 + B04)
# Compute NDVI and append as an extra band.
# Small epsilon (1e-6) prevents division by zero in dark/water pixels.

b08 = composite[S2_BANDS.index("B08")]
b04 = composite[S2_BANDS.index("B04")]
ndvi = (b08 - b04) / (b08 + b04 + 1e-6)
composite = np.concatenate([composite, ndvi[None, ...]], axis=0)
S2_FEATURE_NAMES = S2_BANDS + ["NDVI"]

# Save as a multi-band GeoTIFF with LZW compression and named band descriptions.
S2_TIF = WORK_DIR / "sentinel2.tif"
with rasterio.open(
    S2_TIF, "w", driver="GTiff", height=h, width=w,
    count=len(S2_FEATURE_NAMES), dtype="float32",
    crs=f"EPSG:{crs0}", transform=dst_tfm, compress="lzw",
) as dst:
    for i, name in enumerate(S2_FEATURE_NAMES, start=1):
        dst.write(composite[i-1], i)
        dst.set_band_description(i, name)

print(f"Saved {S2_TIF}   shape={composite.shape}")

## 5. Experiment 1 — Sentinel-2 spectral features only

Establish a baseline by training a Random Forest regressor on Sentinel-2 reflectance alone (10 bands + NDVI). GEDI shots are sampled at their footprint locations, split 80/20 into train and test sets with a fixed random seed, and this split is reused across all three experiments so R² and RMSE values are directly comparable. The trained model is applied pixel-by-pixel across the full grid to produce the first canopy height GeoTIFF.


In [ ]:
"""Sample all bands of a GeoTIFF at a set of WGS84 lon/lat points.

    Coordinates are reprojected to the raster's CRS before sampling so this
    works regardless of the raster's projection.
    Returns an array of shape (N, n_bands).
"""
def sample_raster_at_points(tif_path, lons, lats):
    with rasterio.open(tif_path) as src:
        tf = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        xs, ys = tf.transform(lons, lats)
        return np.array([list(v) for v in src.sample(zip(xs, ys))], dtype=np.float32)

lons = gedi["longitude"].values
lats = gedi["latitude"].values

s2_feat = sample_raster_at_points(S2_TIF, lons, lats)
print(f"S2 feature shape: {s2_feat.shape} ({len(S2_FEATURE_NAMES)} bands)")

y = gedi["rh98"].values.astype(np.float32)
# Drop shots where any S2 band is NaN (e.g. GEDI footprint fell in a cloud gap)
mask_valid = np.all(np.isfinite(s2_feat), axis=1) & np.isfinite(y)
X_s2 = s2_feat[mask_valid]; y_v = y[mask_valid]
lons_v = lons[mask_valid]; lats_v = lats[mask_valid]
print(f"Valid shots after S2 sampling: {len(y_v):,} of {len(y):,}")

idx = np.arange(len(y_v))
# Fixed random seed ensures the same 80/20 split is used across all experiments,
# making R² and RMSE values directly comparable between them.
idx_tr, idx_te = train_test_split(idx, test_size=0.2, random_state=42)
print(f"train: {len(idx_tr):,}   test: {len(idx_te):,}")

# Persist the split to disk so it can be reloaded or shared independently.
pd.DataFrame({"longitude": lons_v[idx_tr], "latitude": lats_v[idx_tr],
              "rh98": y_v[idx_tr]}).to_csv(WORK_DIR / "gedi_train.csv", index=False)
pd.DataFrame({"longitude": lons_v[idx_te], "latitude": lats_v[idx_te],
              "rh98": y_v[idx_te]}).to_csv(WORK_DIR / "gedi_test.csv",  index=False)


In [ ]:
# Hyperparameters are shared across all three experiments.
# min_samples_leaf=20 acts as regularisation: GEDI shots have spatial
# autocorrelation, so large leaves prevent the model memorising local clusters.

RF_PARAMS = dict(
    n_estimators=100,   # number of trees; 100 is sufficient for this feature count
    max_depth=20,       # limits tree depth to reduce overfitting
    min_samples_split=5,
    min_samples_leaf=20,# each leaf must represent at least 20 samples
    max_features=None,  # use all features at each split (sqrt is another common choice)
    random_state=42,
    n_jobs=-1           # use all available CPU cores
    )

def train_eval(X, y, idx_tr, idx_te, feature_names, tag):
    """Train a Random Forest and report train/test R² and RMSE."""
    rf = RandomForestRegressor(**RF_PARAMS)
    rf.fit(X[idx_tr], y[idx_tr])
    yp_tr = rf.predict(X[idx_tr]); yp_te = rf.predict(X[idx_te])
    r2_tr = r2_score(y[idx_tr], yp_tr); r2_te = r2_score(y[idx_te], yp_te)
    rmse_tr = math.sqrt(mean_squared_error(y[idx_tr], yp_tr))
    rmse_te = math.sqrt(mean_squared_error(y[idx_te], yp_te))
    print(f"\n[{tag}]  n_features={X.shape[1]}")
    print(f"  Train R²={r2_tr:.4f}   RMSE={rmse_tr:.3f} m")
    print(f"  Test  R²={r2_te:.4f}   RMSE={rmse_te:.3f} m   (gap={r2_tr-r2_te:+.3f})")
    # Feature importances show which bands contribute most to height predictions
    imp = pd.Series(rf.feature_importances_, index=feature_names)\
            .sort_values(ascending=False)
    print("  Top 5 features:")
    print(imp.head(5).to_string())
    return rf, dict(tag=tag, n_features=X.shape[1],
                    r2_train=r2_tr, r2_test=r2_te,
                    rmse_train_m=rmse_tr, rmse_test_m=rmse_te)

rf1, res1 = train_eval(X_s2, y_v, idx_tr, idx_te, S2_FEATURE_NAMES, "S2 only")


In [ ]:
"""Apply a trained model pixel-by-pixel across stacked feature rasters.

    All feature rasters are reprojected to the grid of the first TIF before
    prediction. Output is clipped to 0-80 m (physically plausible canopy range)
    and saved as a single-band float32 GeoTIFF.
"""
def predict_to_geotiff(model, feature_tifs, feature_names, out_path):
    with rasterio.open(feature_tifs[0]) as src0:
        prof = src0.profile.copy(); h, w = src0.height, src0.width
        crs0 = src0.crs; tfm0 = src0.transform
        base = src0.read().astype("float32")
    stacks = [base]
    # Reproject additional feature rasters onto the reference grid
    for tif in feature_tifs[1:]:
        with rasterio.open(tif) as src:
            arr = np.zeros((src.count, h, w), dtype="float32")
            for i in range(src.count):
                reproject(
                    source=src.read(i+1), destination=arr[i],
                    src_transform=src.transform, src_crs=src.crs,
                    dst_transform=tfm0, dst_crs=crs0,
                    resampling=Resampling.bilinear,
                )
            stacks.append(arr)
    data = np.concatenate(stacks, axis=0)
    assert data.shape[0] == len(feature_names), \
        f"Got {data.shape[0]} bands but expected {len(feature_names)} features"
    # Flatten to (n_pixels, n_features) for sklearn, then predict only valid pixels
    flat = data.reshape(data.shape[0], -1).T
    valid = np.all(np.isfinite(flat), axis=1)
    pred = np.full(flat.shape[0], np.nan, dtype=np.float32)
    pred[valid] = model.predict(flat[valid])
    pred = np.clip(pred, 0, 80).reshape(h, w) # clip to plausible canopy range
    prof.update(count=1, dtype="float32", compress="lzw", tiled=True, blockxsize=256, blockysize=256)
    with rasterio.open(out_path, "w", **prof) as dst:
        dst.write(pred.astype("float32"), 1)
    return pred

pred1 = predict_to_geotiff(rf1, [S2_TIF], S2_FEATURE_NAMES,
                           WORK_DIR / "chm_exp1_s2only.tif")

plt.figure(figsize=(8, 6))
im = plt.imshow(pred1, cmap="viridis", vmin=0, vmax=40)
plt.colorbar(im, label="Canopy height (m)")
plt.title("Experiment 1 — predicted canopy height (S2 only)")
plt.axis("off"); plt.show()


## 6. Sentinel-1 SAR — VV / VH summer median composite

Build a Sentinel-1 radar composite to complement the optical signal. Because C-band microwave pulses penetrate cloud cover, SAR provides observations on days when Sentinel-2 would be obscured. The Radiometrically Terrain Corrected (RTC) collection from Microsoft Planetary Computer is used; VV and VH polarisation scenes are median-composited across summer 2024 and converted from linear backscatter to **decibels** (`10·log₁₀(linear)`), which compresses the dynamic range and makes the values better-behaved as ML features.


In [ ]:
search = cat.search(
    collections=["sentinel-1-rtc"],
    bbox=bbox,
    datetime="2024-06-01/2024-09-30",
)
# Sentinel-1 RTC (Radiometrically Terrain Corrected) removes geometric
# distortions caused by terrain. Limiting to 10 scenes caps memory usage.
items_s1 = list(search.items())[:10]
print(f"Sentinel-1 RTC scenes used: {len(items_s1)}")

def _read_s1_band(item, asset, dst_tfm, dst_crs, h, w):
    """Read a single Sentinel-1 polarisation band and reproject to target grid."""
    href = item.assets[asset].href
    out = np.full((h, w), np.nan, dtype="float32")
    with rasterio.open(href) as src:
        reproject(
            source=rasterio.band(src, 1), destination=out,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=dst_tfm, dst_crs=dst_crs,
            resampling=Resampling.bilinear,
        )
    return out

vv_stack, vh_stack = [], []
for it in items_s1:
    try:
        vv_stack.append(_read_s1_band(it, "vv", dst_tfm, f"EPSG:{crs0}", h, w))
        vh_stack.append(_read_s1_band(it, "vh", dst_tfm, f"EPSG:{crs0}", h, w))
    except Exception as e:
        print(f"  ⚠ {e}")

# Convert linear backscatter to decibels after taking the median across scenes.
# Clipping to 1e-6 avoids log(0). dB values are better-behaved as ML features
# because linear backscatter spans several orders of magnitude.
vv = 10 * np.log10(np.clip(np.nanmedian(np.stack(vv_stack, 0), axis=0), 1e-6, None))
vh = 10 * np.log10(np.clip(np.nanmedian(np.stack(vh_stack, 0), axis=0), 1e-6, None))

S1_TIF = WORK_DIR / "sentinel1.tif"
S1_FEATURE_NAMES = ["VV_dB", "VH_dB"]
with rasterio.open(
    S1_TIF, "w", driver="GTiff", height=h, width=w, count=2,
    dtype="float32", crs=f"EPSG:{crs0}", transform=dst_tfm, compress="lzw",
) as dst:
    dst.write(vv.astype("float32"), 1); dst.set_band_description(1, "VV_dB")
    dst.write(vh.astype("float32"), 2); dst.set_band_description(2, "VH_dB")
print(f"Saved {S1_TIF}")


## 7. Experiment 2 — Sentinel-2 + Sentinel-1

Re-train the Random Forest with the two SAR bands (VV_dB, VH_dB) appended to the Sentinel-2 feature set. Hyperparameters are held constant so any change in test R² or RMSE is attributable solely to the additional radar features. SAR backscatter is sensitive to canopy structure and moisture content, which are largely orthogonal to optical reflectance.


In [ ]:
s1_feat = sample_raster_at_points(S1_TIF, lons, lats)[mask_valid]

X_s2s1 = np.concatenate([X_s2, s1_feat], axis=1)
FEAT_S2S1 = S2_FEATURE_NAMES + S1_FEATURE_NAMES

rf2, res2 = train_eval(X_s2s1, y_v, idx_tr, idx_te, FEAT_S2S1, "S2 + S1")

pred2 = predict_to_geotiff(
    rf2, [S2_TIF, S1_TIF], FEAT_S2S1,
    WORK_DIR / "chm_exp2_s2_s1.tif",
)
plt.figure(figsize=(8, 6))
plt.imshow(pred2, cmap="viridis", vmin=0, vmax=40)
plt.colorbar(label="Canopy height (m)")
plt.title("Experiment 2 — predicted canopy height (S2 + S1)")
plt.axis("off"); plt.show()


## 8. COP30 DEM — elevation, slope, aspect

Download the 30 m Copernicus DEM via the **OpenTopography** API and derive topographic features: elevation, slope (degrees), and aspect. Aspect is encoded as `sin(aspect)` and `cos(aspect)` rather than a raw angle so that 0° and 360° map to the same value, avoiding a false discontinuity at north. All bands are reprojected to the Sentinel-2 30 m grid before saving.


In [ ]:
import os
import requests

# Obtain a free OpenTopography API key from https://www.opentopography.org

# ── Optional: Secure Credential Handling via Colab Secrets ───────────────────
# In Google Colab: click the 🔑 key icon in the left sidebar → add a secret
# named OPENTOPOGRAPHY_API_KEY with your key → enable Notebook access.
# Running locally? Skip this block — fill in the key directly below instead.

try:
    from google.colab import userdata
    _ot_key = userdata.get("OPENTOPOGRAPHY_API_KEY")
    if _ot_key:
        OPENTOPOGRAPHY_API_KEY = _ot_key
        print("✓ OpenTopography API key loaded from Colab Secrets.")
    else:
        print("⚠ Secret found but OPENTOPOGRAPHY_API_KEY is empty. "
              "Falling back to the value set below.")
        OPENTOPOGRAPHY_API_KEY = None
except Exception:
    # Not running in Colab, or secret key doesn't exist yet.
    OPENTOPOGRAPHY_API_KEY = None

# ── Hardcoded fallback ────────────────────────────────────────────────────────
# Fill this in if you are NOT using Colab Secrets (see above).
if OPENTOPOGRAPHY_API_KEY is None:
    OPENTOPOGRAPHY_API_KEY = ""   # e.g. "abcd1234..."

assert OPENTOPOGRAPHY_API_KEY, (
    "Please set OPENTOPOGRAPHY_API_KEY before running — either via Colab "
    "Secrets (recommended) or by editing this cell directly. "
    "Get one (free) at https://portal.opentopography.org."
)
os.environ["OPENTOPOGRAPHY_API_KEY"] = OPENTOPOGRAPHY_API_KEY


OT_URL = "https://portal.opentopography.org/API/globaldem"
params = dict(demtype="COP30", south=bbox[1], north=bbox[3],
              west=bbox[0], east=bbox[2], outputFormat="GTiff",
              API_Key=os.environ["OPENTOPOGRAPHY_API_KEY"])
print("Downloading COP30 DEM...")
r = requests.get(OT_URL, params=params, timeout=300)
r.raise_for_status()
DEM_RAW = WORK_DIR / "cop30_raw.tif"
DEM_RAW.write_bytes(r.content)

# Pixel size in metres depends on CRS. For geographic CRS (degrees), convert
# using the cosine correction for longitude and a fixed 111 km/degree for latitude.
with rasterio.open(DEM_RAW) as src:
    dem = src.read(1).astype("float32")
    if src.crs.is_geographic:
        clat = 0.5 * (src.bounds.top + src.bounds.bottom)
        res_y_m = src.res[1] * 111_000.0
        res_x_m = src.res[0] * 111_000.0 * np.cos(np.radians(clat))
    else:
        res_x_m, res_y_m = src.res
    # np.gradient returns rise/run in pixel units; dividing by resolution
    # converts to rise/run in metres before computing the angle.
    grad_y, grad_x = np.gradient(dem)
    slope = np.degrees(np.arctan(np.sqrt((grad_y/res_y_m)**2 + (grad_x/res_x_m)**2)))
    # arctan2 returns angle from east; convert to compass bearing (0° = north).
    aspect_rad = np.arctan2(-grad_y, grad_x)
    aspect = (90.0 - np.degrees(aspect_rad)) % 360.0
    raw_tfm = src.transform; raw_crs = src.crs

# Reproject each band to the S2 grid
def _reproject_to_s2(arr, src_tfm, src_crs):
    out = np.full((h, w), np.nan, dtype="float32")
    reproject(source=arr, destination=out,
              src_transform=src_tfm, src_crs=src_crs,
              dst_transform=dst_tfm, dst_crs=f"EPSG:{crs0}",
              resampling=Resampling.bilinear)
    return out

elev_g  = _reproject_to_s2(dem,  raw_tfm, raw_crs)
slope_g = _reproject_to_s2(slope.astype("float32"), raw_tfm, raw_crs)
asp_g   = _reproject_to_s2(aspect.astype("float32"), raw_tfm, raw_crs)

# Encode aspect as sin/cos so that 0° and 360° map to the same point,
# avoiding a false discontinuity at north that would confuse the model.
asp_sin = np.sin(np.radians(asp_g))
asp_cos = np.cos(np.radians(asp_g))

TOPO_TIF = WORK_DIR / "topography.tif"
TOPO_FEATURE_NAMES = ["elevation", "slope_deg", "aspect_sin", "aspect_cos"]
with rasterio.open(
    TOPO_TIF, "w", driver="GTiff", height=h, width=w, count=4,
    dtype="float32", crs=f"EPSG:{crs0}", transform=dst_tfm, compress="lzw",
) as dst:
    for i, (arr, name) in enumerate(zip([elev_g, slope_g, asp_sin, asp_cos],
                                        TOPO_FEATURE_NAMES), 1):
        dst.write(arr.astype("float32"), i)
        dst.set_band_description(i, name)
print(f"Saved {TOPO_TIF}")
print(f"Elevation: {np.nanmin(elev_g):.0f} – {np.nanmax(elev_g):.0f} m")
print(f"Slope:     median {np.nanmedian(slope_g):.1f}°, max {np.nanmax(slope_g):.1f}°")

## 9. Experiment 3 — Sentinel-2 + Sentinel-1 + Topography

Train the full 17-feature model combining all three data sources: 11 Sentinel-2 optical features, 2 Sentinel-1 SAR features, and 4 topographic features (elevation, slope, aspect sin/cos). Topography conditions where forests grow and how tall they get independently of seasonal spectral signals, so it is expected to reduce residual error in areas of strong relief.


In [ ]:
topo_feat = sample_raster_at_points(TOPO_TIF, lons, lats)[mask_valid]

X_all = np.concatenate([X_s2, s1_feat, topo_feat], axis=1)
FEAT_ALL = S2_FEATURE_NAMES + S1_FEATURE_NAMES + TOPO_FEATURE_NAMES

rf3, res3 = train_eval(X_all, y_v, idx_tr, idx_te, FEAT_ALL, "S2 + S1 + Topo")

pred3 = predict_to_geotiff(
    rf3, [S2_TIF, S1_TIF, TOPO_TIF], FEAT_ALL,
    WORK_DIR / "chm_exp3_s2_s1_topo.tif",
)
plt.figure(figsize=(8, 6))
plt.imshow(pred3, cmap="viridis", vmin=0, vmax=40)
plt.colorbar(label="Canopy height (m)")
plt.title("Experiment 3 — predicted canopy height (S2 + S1 + Topo)")
plt.axis("off"); plt.show()


## 10. Comparison

Summarise R² and RMSE across all three experiments for both the training set and the held-out test set. The train/test gap column indicates overfitting — a large gap means the model has memorised training patterns that do not generalise. Improvements in test R² from one experiment to the next reflect the genuine predictive contribution of each added data source.


In [ ]:
summary = pd.DataFrame([res1, res2, res3])
summary["gap_r2"] = (summary["r2_train"] - summary["r2_test"]).round(4)
cols = ["tag", "n_features",
        "r2_train", "r2_test", "gap_r2",
        "rmse_train_m", "rmse_test_m"]
print(summary[cols].to_string(index=False))
summary.to_csv(WORK_DIR / "experiment_summary.csv", index=False)


### Outputs

All artifacts are written under `./canopy_workflow/`:

- `gedi_shots.csv`, `gedi_train.csv`, `gedi_test.csv` — GEDI labels
- `sentinel2.tif`, `sentinel1.tif`, `topography.tif` — feature rasters
- `chm_exp1_s2only.tif`, `chm_exp2_s2_s1.tif`, `chm_exp3_s2_s1_topo.tif` — predicted canopy-height GeoTIFFs
- `experiment_summary.csv` — final comparison table

Each `.tif` is single-band float32, canopy height in metres, clipped to 0–80 m,
aligned to the Sentinel-2 30 m grid in its native UTM zone. They can be
viewed in QGIS, ArcGIS, or any rasterio-aware Python session.
